In [1]:
import re
import os
import warnings
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

python-dotenv could not parse statement starting at line 7


True

In [2]:
def clean_text(text:str):
    text = re.sub("-\n","",text)
    text = re.sub("  ","",text)

    text = re.sub(r"[ \t]+", " ", text)

    lines = text.split("\n")
    lines = [line for line in lines if not re.fullmatch(r"\s*\d{1,4}\s*", line)]
    text = "\n".join(lines)

    text = text.strip()
    return text

result = clean_text("""This        is      the        first          line                  with               lots                  of                        spaces
And		  this		     is		     the		     second		     line		     with		     generous		     tabs""")
print(result)

Thisisthefirstlinewith lotsofspaces
And this is the second line with generous tabs


In [3]:
from tqdm import tqdm
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

def extract_clean_text(docs):
    pages = []

    for page_num, page in enumerate(docs):
        if page_num >= 20:
            text = page.page_content

            # Remove multiple spaces
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r"\n+", " ", text)

            # Fix paragraph breaks (optional improvement)
            text = re.sub(r'\.\s+', '.\n', text)
            #text = re.sub(r"\.\n+", " ", text)
            text = re.sub(r"Fig.", " ", text)
            pages.append(
                Document(
                    page_content=text.strip(),
                    metadata=page.metadata,
                )
            )

    return pages


loader = PyMuPDFLoader(r"D:\Final_Year\data\book_3.pdf")
documents = loader.load()
documents = extract_clean_text(documents)


In [4]:
len(documents)

365

In [5]:
documents[0].metadata

{'producer': 'Adobe PDF Library 7.0',
 'creator': 'Adobe Acrobat 6.0',
 'creationdate': '2016-01-19T01:23:56+00:00',
 'source': 'D:\\Final_Year\\data\\book_3.pdf',
 'file_path': 'D:\\Final_Year\\data\\book_3.pdf',
 'total_pages': 385,
 'format': 'PDF 1.6',
 'title': 'Endoscopic Pituitary Surgery',
 'author': 'Anand, Vijay K.,Schwartz, Theodore H.,American Association of Neurosurgeons.',
 'subject': '',
 'keywords': '',
 'moddate': '2016-01-19T13:34:53+00:00',
 'trapped': '',
 'modDate': 'D:20160119133453Z',
 'creationDate': 'D:20160119012356Z',
 'page': 20}

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
semantic_encoder = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
chunker = SemanticChunker(
    semantic_encoder,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)
documents = chunker.split_documents(documents)
documents

In [18]:
print(len(documents))
print(type(documents))

1652
<class 'list'>


In [19]:
texts = [chunk.page_content for chunk in documents]
print(len(texts))

embeddings_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")


embeddings_text = embeddings_model.encode(texts, show_progress_bar=True)
embeddings_text


1652


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

array([[ 0.27545884, -0.3441546 , -0.3669956 , ...,  0.38380313,
         0.2390515 ,  0.1991375 ],
       [ 0.07640245, -0.6986943 ,  0.23820548, ...,  0.44894472,
         0.57463336, -0.31169558],
       [ 0.10415162, -0.31918243,  0.36163598, ...,  0.48692366,
         0.23531313, -0.38946924],
       ...,
       [-0.00891036, -0.17044038,  0.47187138, ...,  0.47025833,
         0.25301206, -0.21036947],
       [ 0.1325127 , -0.27755585,  0.14751294, ...,  0.18260005,
         0.3237785 , -0.14862612],
       [-0.6692885 ,  0.5254556 ,  0.34783003, ...,  0.09515939,
         0.48778322, -0.54615587]], shape=(1652, 768), dtype=float32)

In [23]:
print(embeddings_text.shape)
print(embeddings_model.get_sentence_embedding_dimension())

(1652, 768)
768


In [29]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

qdrant_client = QdrantClient(
    url= os.getenv("QDRANT_URI"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print(qdrant_client.get_collections())

collections=[CollectionDescription(name='Adaptive_RAG')]


In [37]:
collection_name = 'Adaptive_RAG'
if not qdrant_client.collection_exists(collection_name=collection_name):
    # 3. If it does not exist, create it
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")

Collection 'Adaptive_RAG' already exists.


In [38]:
qdrant_client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='Adaptive_RAG')])

In [40]:
print(type(embeddings_text))
print(len(embeddings_text))
print(len(embeddings_text[0]))

<class 'numpy.ndarray'>
1652
768


In [41]:
info = qdrant_client.get_collection(collection_name)
print(info.config.params.vectors.size)

768


In [43]:
from qdrant_client.models import PointStruct

def vector_store(documents, embeddings, batch_size):
    total_length = len(documents)

    for start in range(0, total_length, batch_size):
        end = min(start + batch_size, total_length)
        batch_docs = documents[start:end]
        batch_embeddings = embeddings[start:end]

        points = []

        for i,(doc, embed) in enumerate(zip(batch_docs, batch_embeddings)):
            doc_id = start+i
            metadata = dict(doc.metadata)
            metadata["doc_index"] = start + i
            metadata["content_length"] = len(doc.page_content)
            metadata["text"] = doc.page_content

            points.append(
                PointStruct(
                    id=doc_id,
                    vector=embed.tolist(),
                    payload=metadata
                )
            )

        qdrant_client.upsert(collection_name=collection_name,points=points)
        print(f"Uploaded batch {start} to {end}")
vector_store(documents, embeddings_text,batch_size=500)
print("Ingestion Complete")

Uploaded batch 0 to 500
Uploaded batch 500 to 1000
Uploaded batch 1000 to 1500
Uploaded batch 1500 to 1652
Ingestion Complete


In [65]:
query_embeddings = embeddings_model.encode(["Surgical Indications for the Treatment of Prolactinomas"])

results = qdrant_client.query_points(
    collection_name=collection_name,
    query=query_embeddings[0].tolist(),
    limit=5)

In [73]:
for i in results.points:
    print(i.payload['text'])

9 Prolactinomas and Apoplexy 91 Surgical Indications for the Treatment of Prolactinomas The surgical indications for the treatment of prolactinomas take into account the need to normalize the serum prolactin levels; to decompress the visual apparatus, cranial nerves, and the rest of the pituitary gland; and to prevent the tu- mor from progressing or recurring. The surgical strategy used differs depending on whether a surgical cure can be expected or the surgery is adjuvant therapy. Surgery as a Curative Therapy In cases of microprolactinomas that are restricted to the pi- tuitary gland, surgery is an option for patients who fail med- ical treatment by remaining hyperprolactinemic. About 15% of patients undergoing medical treatment do not respond to either bromocriptine or cabergoline, with a significant decrease in prolactin levels by 6 weeks after initiation of medical therapy. Surgery is also warranted for patients who do not tolerate medical therapy because of its side effects or fo

In [57]:
print(type(results))

<class 'qdrant_client.http.models.models.QueryResponse'>
